In [1]:
import pandas as pd
import json

In [21]:
df = pd.read_csv('hgnc_feb2026.txt', sep='\t', dtype={'OMIM ID': str, 'NCBI gene ID': str, 'Chromosome': str})
# limit to autosomes and protein-coding genes only
excluded = ['X', 'Y', 'X and Y', 'not on reference assembly', 'mitochondria', 'reserved', 'unplaced']
df = df[(~df['Chromosome'].isin(excluded)) & (df['Locus group']=='protein-coding gene')].reset_index(drop=True)
df = df.drop(columns=['Locus group'])
df['HGNC ID'] = df['HGNC ID'].str.replace('HGNC:', '', regex=False)

In [23]:
cols = ['Previous symbol', 'Alias symbol', 'HGNC ID', 'NCBI gene ID', 'UCSC gene ID', 'Ensembl gene ID', 'RefSeq accession', 'CCDS accession', 'OMIM ID']

jumbo_dict = {k: {} for k in ['Symbol', *cols[2:]]}

approved_symbols = df['Approved symbol'].unique()

for asym in approved_symbols:
    jumbo_dict['Symbol'][asym] = asym
    asym_df = df[df['Approved symbol'] == asym]
    for col in cols:
        key = 'Symbol' if col in ['Previous symbol', 'Alias symbol'] else col
        col_vals = asym_df[col].dropna().unique()
        for val in col_vals:
            jumbo_dict[key][val.strip()] = asym

In [24]:
with open('HGNC_dicts.json', 'w') as f:
    json.dump(jumbo_dict, f)